## 🎬 Movie Reviews using NLP 📽️🍿
In this lab, we will build a text classification system using Natural Language Processing (NLP) techniques to classify movie reviews as either positive or negative. 🧠📝

🔍 What We’ll Cover:
- Data Loading
- Data Splitting & Vectorization
- Model Building & Training
- Model Evaluation
- Testing Reviews

💡 NOTE: Please pay close attention to the explanation during the lab, as it will help you answer the Golden Question 🏆 at the end.


### 🛠️ Import library

In [1]:
!pip install datasets scikit-learn textblob -q
!python -m textblob.download_corpora

Finished.


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\hayaa\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is alr

In [2]:
# colab
#!pip install gensim

#!pip install --upgrade gensim


In [3]:
# vs code
%pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

from textblob import TextBlob 

import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.porter import PorterStemmer

In [5]:
#import nltk
#nltk.download("stopwords")

### 🔤📄 Data Loading 

In [6]:
dataset = pd.read_csv('movie_data.csv', skiprows=[35810])

dataset.head()

,review,sentiment
0,I went and saw this movie last night after bei...,1
1,Actor turned director Bill Paxton follows up h...,1
2,As a recreational golfer with some knowledge o...,1
3,"I saw this film in a sneak preview, and it is ...",1
4,Bill Paxton has taken the true story of the 19...,1


### 🕵️‍♀️ How can I know the number of samples in the dataset?

In [7]:
# TO-Do
dataset.shape

(49999, 2)

### 📐🧮 Data Splitting & Vectorization

In [8]:
english_stopwords = stopwords.words('english')
stemmer = PorterStemmer()

# Define cleaning function 
def clean_review(text):
  # Tokenize the text and filter out non-alphabetic words
  words = [word for word in word_tokenize(text) if word.isalpha()] 

  # Remove stopwords and stem the remaining words
  cleaned_words = [stemmer.stem(word) for word in words if word not in english_stopwords]

  # Join the cleaned words into a single string
  text = ' '.join(cleaned_words)

  return text

In [9]:
dataset['review_after_cleaning'] = dataset['review'].apply(clean_review)

In [10]:
# Sample corpus
sentences = [
    "I love this movie",
    "It was bad"
]

# Tokenize each sentence
tokenized = [word_tokenize(sent.lower()) for sent in sentences] # ---> [['i', 'love', 'this', 'movie'], ['it', 'was', 'bad']]

# Train Word2Vec model
model = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=1) 

# Get vector for a word
vector = model.wv['movie']  # vector for the word "learning"
print("Vector for 'movie':", vector) 

# Find similar words
print("Similar to 'movie':", model.wv.most_similar('movie'))

Vector for 'movie': [-8.2426779e-03  9.2993546e-03 -1.9766092e-04 -1.9672764e-03
  4.6036304e-03 -4.0953159e-03  2.7431143e-03  6.9399667e-03
  6.0654259e-03 -7.5107943e-03  9.3823504e-03  4.6718083e-03
  3.9661205e-03 -6.2435055e-03  8.4599797e-03 -2.1501649e-03
  8.8251876e-03 -5.3620026e-03 -8.1294188e-03  6.8245591e-03
  1.6711927e-03 -2.1985089e-03  9.5136007e-03  9.4938548e-03
 -9.7740470e-03  2.5052286e-03  6.1566923e-03  3.8724565e-03
  2.0227872e-03  4.3050171e-04  6.7363144e-04 -3.8206363e-03
 -7.1402504e-03 -2.0888723e-03  3.9238976e-03  8.8186832e-03
  9.2591504e-03 -5.9759365e-03 -9.4026709e-03  9.7643770e-03
  3.4297847e-03  5.1661171e-03  6.2823449e-03 -2.8042626e-03
  7.3227035e-03  2.8302716e-03  2.8710044e-03 -2.3803699e-03
 -3.1282497e-03 -2.3701417e-03  4.2764368e-03  7.6057913e-05
 -9.5842788e-03 -9.6655441e-03 -6.1481940e-03 -1.2856961e-04
  1.9974159e-03  9.4319675e-03  5.5843508e-03 -4.2906962e-03
  2.7831673e-04  4.9643586e-03  7.6983096e-03 -1.1442233e-03
  4.

In [11]:
text = dataset['review_after_cleaning'].values
label = dataset['sentiment'].values 

X_train, X_test, y_train, y_test = train_test_split(text, label, train_size=0.5, test_size=0.5, random_state=42)
    
vectorizer = CountVectorizer(binary=True, max_features=10000) 

X_train_vec = vectorizer.fit_transform(X_train)

X_test_vec = vectorizer.transform(X_test) 

### 🌐🤖 Model Building & Evaluation

In [12]:
model = LogisticRegression()

model.fit(X_train_vec,y_train)

acc_train = model.score(X_train_vec,y_train)
print('Training Accuracy:', acc_train)

Training Accuracy: 0.9809592383695348


In [13]:
y_pred = model.predict(X_test_vec)

accuracy = model.score(X_test_vec,y_pred)
print("✅ Accuracy: ", accuracy)

✅ Accuracy:  1.0


### The GOLDEN Question 🏆:
If we can already perform tasks like classification using logistic regression by calculating labels and predictions, then why do we need NLP?


### 🔍🎭 Testing

In [14]:
def correct_text(text):
    return str(TextBlob(text).correct())


def predict(model, vectorizer, review):
    review = correct_text(review)       
    review = clean_review(review)     
    review_bow = vectorizer.transform([review])
    sentiment = 'Positive 😊' if model.predict(review_bow)[0] == 1 else 'Negative 😞'
    return sentiment


In [15]:
review = 'The movie was great!'
predict(model, vectorizer, review)

'Positive 😊'

In [16]:
review = 'It was boring'
predict(model, vectorizer, review)

'Negative 😞'

## 📌👏 Task:

Change the `review` to try your own movie reviews 🎥🍿, and let the model classify it as a positive 😊 or negative 😞 review.

In [ ]:
review = '' # To-Do
predict(model, vectorizer, review)